In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time
from pymongo import MongoClient

# --- 1. Configuración de Chrome ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"   # ruta dentro del contenedor
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

# Iniciar el driver con webdriver_manager
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# --- 2. Abrir ODEPA ---
driver.get("https://www.odepa.gob.cl/precios/consumidor")

print("ACCIÓN REQUERIDA:")
print("1. Ve al navegador en el contenedor (VNC).")
print("2. Resuelve el captcha manualmente.")
print("3. Aplica los filtros (ej: producto=Naranja, región=Coquimbo, periodo=Mensual).")
input("Cuando veas la tabla de resultados, presiona ENTER aquí...")

# --- 3. Extraer la tabla ---
time.sleep(5)
productos = []
filas = driver.find_elements(By.CSS_SELECTOR, "table tr")

for fila in filas[1:]:
    columnas = [c.text for c in fila.find_elements(By.TAG_NAME, "td")]
    if columnas and len(columnas) >= 8:
        productos.append({
            "mes_año": columnas[0],
            "producto": columnas[1],
            "variedad": columnas[2],
            "calidad": columnas[3],
            "unidad": columnas[4],
            "precio_minimo": columnas[5],
            "precio_maximo": columnas[6],
            "precio_promedio": columnas[7]
        })

print("Primeros registros extraídos:")
print(productos[:5])

# --- 4. Guardar en MongoDB ---
client = MongoClient("mongodb://bigdata_mongodb:27017/")
db = client["semana6"]
collection = db["odepa_scraping"]

if productos:
    collection.insert_many(productos)
    print(f"{len(productos)} registros guardados en MongoDB")
else:
    print("No se encontraron datos para guardar")

# --- 5. Verificación ---
for doc in collection.find().limit(5):
    print(doc)


ACCIÓN REQUERIDA:
1. Ve al navegador en el contenedor (VNC).
2. Resuelve el captcha manualmente.
3. Aplica los filtros (ej: producto=Naranja, región=Coquimbo, periodo=Mensual).


Cuando veas la tabla de resultados, presiona ENTER aquí... 


Primeros registros extraídos:
[]
No se encontraron datos para guardar
